In [3]:
import json
import numbers
import os
import re
import time
from tabnanny import check

import requests
from numpy import add

mainurl = "https://pokeapi.co/api/v2/"


In [5]:

url = mainurl + "move/1"
response = requests.get(url)
print(response.json()["meta"].keys())


dict_keys(['ailment', 'ailment_chance', 'category', 'crit_rate', 'drain', 'flinch_chance', 'healing', 'max_hits', 'max_turns', 'min_hits', 'min_turns', 'stat_chance'])


In [58]:
url = mainurl + "ability/1"
response = requests.get(url)
print(response.json()["name"].keys())

AttributeError: 'str' object has no attribute 'keys'

In [ ]:
import stat


filename = "pokemon.json"
with open(filename, "r", encoding="utf-8") as f:
    pokemon_dict = json.load(f)
filename = "move.json"
with open(filename, "r", encoding="utf-8") as f:
    try:
        move_dict = json.load(f)
    except json.JSONDecodeError:
        # 万が一ファイルが空っぽなどで壊れていた場合の保険
        move_dict = dict()
for id in pokemon_dict:
    for moveid in list(pokemon_dict[id]["moves"]):
        if str(moveid) not in move_dict:
            url = mainurl + "move/" + str(moveid)
            response = requests.get(url)
            effect_entrys = response.json()["effect_entries"]
            meta = response.json()["meta"]

            for effect_entry in effect_entrys:
                if effect_entry["language"]["name"] == "en":
                    short_effect = effect_entry["short_effect"]
                    break
            stat_changes = response.json()["stat_changes"]
            if len(stat_changes) == 0:
                stat_changes_value = 0
                stat_changes_stat = ""
            else:
                stat_changes_value = stat_changes[0]["change"]
                stat_changes_stat = stat_changes[0]["stat"]["name"]
            names = response.json()["names"]
            for name in names:
                if name["language"]["name"] == "ja-hrkt":
                    jpname = name["name"]
                    break
                else:
                    print(f"{response.json()['name']} name was not found")
                    jpname = ""
            if type(meta) is dict:
                move_dict[str(moveid)] = {
                    "id": moveid,
                    "name": response.json()["name"],
                    "type": response.json()["type"]["name"],
                    "power": response.json()["power"],
                    "accuracy": response.json()["accuracy"],
                    "pp": response.json()["pp"],
                    "damage_class": response.json()["damage_class"]["name"],
                    "effect": short_effect,
                    "effect_chance": response.json()["effect_chance"],
                    "stat_changes_value": stat_changes_value,
                    "stat_changes_stat": stat_changes_stat,
                    "jpname": jpname,
                    # meta
                    "ailment": meta["ailment"]["name"],
                    "ailment_chance": meta["ailment_chance"],
                    "category": meta["category"]["name"],
                    "crit_rate": meta["crit_rate"],
                    "flinch_chance": meta["flinch_chance"],
                    "healing": meta["healing"],
                    "min_hits": meta["min_hits"],
                    "max_hits": meta["max_hits"],
                    "min_turns": meta["min_turns"],
                    "max_turns": meta["max_turns"],
                    "stat_chance": meta["stat_chance"],
                    "drain": meta["drain"],
                }
                print(f"追加：ID {moveid} - {move_dict[str(moveid)]['name']} exist meta")
            else:
                move_dict[str(moveid)] = {
                    "id": moveid,
                    "name": response.json()["name"],
                    "type": response.json()["type"]["name"],
                    "power": response.json()["power"],
                    "accuracy": response.json()["accuracy"],
                    "pp": response.json()["pp"],
                    "damage_class": response.json()["damage_class"]["name"],
                    "effect": short_effect,
                    "effect_chance": response.json()["effect_chance"],
                    "stat_changes_value": stat_changes_value,
                    "stat_changes_stat": stat_changes_stat,
                    "jpname": jpname,
                }
                print(f"追加：ID {moveid} - {move_dict[str(moveid)]['name']} except meta")
        else:
            print(f"スキップ：ID {moveid}")

        # save
    with open(filename, "w", encoding="utf-8") as f:
        f.write(json.dumps(move_dict, indent=4))
    print(f"finish {pokemon_dict[id]["name"]}")


In [4]:
# sorting
filename = "JSON/ability.json"
with open(filename, "r", encoding="utf-8") as f:
    move_dict = json.load(f)
move_dict = dict(sorted(move_dict.items(), key=lambda item: int(item[0])))
with open(filename, "w", encoding="utf-8") as f:
    f.write(json.dumps(move_dict, indent=4, ensure_ascii=False))

In [ ]:


filename = "pokemon.json"
with open(filename, "r", encoding="utf-8") as f:
    pokemon_dict = json.load(f)
for id in pokemon_dict:
    id = int(id)
    url = mainurl + "pokemon/" + str(id)
    response = requests.get(url)
    weight = response.json()["weight"]
    height = response.json()["height"]
    pokemon_dict[str(id)]["weight"] = weight
    pokemon_dict[str(id)]["height"] = height
    print(f"追加：ID {id} - {pokemon_dict[str(id)]['name']}")
with open(filename, "w", encoding="utf-8") as f:
    f.write(json.dumps(pokemon_dict, indent=4, ensure_ascii=False))

In [ ]:
filename = "pokemon.json"
with open(filename, "r", encoding="utf-8") as f:
    pokemon_dict = json.load(f)

filename = "ability.json"
with open(filename, "r", encoding="utf-8") as f:
    try:
        ability_dict = json.load(f)
    except json.JSONDecodeError:
        ability_dict = {}
for id in pokemon_dict:
    abilities = pokemon_dict[id]["abilities"]  
    for ability in abilities:
        id = int(ability["ability_id"])
        url = mainurl + "ability/" + str(id)
        response = requests.get(url)
        effect_entries = response.json()["effect_entries"]
        for effect_entry in effect_entries:
            if effect_entry["language"]["name"] == "en":
                effect = effect_entry["effect"]
                short_effect = effect_entry["short_effect"]
                break
        natural_name = response.json()["name"]
        names = response.json()["names"]
        for name in names:
            if name["language"]["name"] == "ja-hrkt":
                jpname = name["name"]
                break
            elif name["language"]["name"] == "ja":
                jpname = name["name"]
                break
            else:
                jpname = ""
        ability_dict[id] = {
            "effect": effect,
            "short_effect": short_effect,
            "name": natural_name,
            "jpname": jpname
        }
        print(f"add {jpname},{natural_name}{effect}")
with open(filename, "w", encoding="utf-8") as f:
    f.write(json.dumps(ability_dict, indent=4, ensure_ascii=False))

add しんりょく,overgrowWhen this Pokémon has 1/3 or less of its HP remaining, its Grass-type moves inflict 1.5× as much regular damage.
add ようりょくそ,chlorophyllThis Pokémon's Speed is doubled during strong sunlight.

This bonus does not count as a stat modifier.
add もうか,blazeWhen this Pokémon has 1/3 or less of its HP remaining, its Fire-type moves inflict 1.5× as much regular damage.
add サンパワー,solar-powerDuring strong sunlight, this Pokémon has 1.5× its Special Attack but takes 1/8 of its maximum HP in damage after each turn.
add げきりゅう,torrentWhen this Pokémon has 1/3 or less of its HP remaining, its Water-type moves inflict 1.5× as much regular damage.
add あめうけざら,rain-dishThis Pokémon heals for 1/16 of its maximum HP after each turn during rain.
add むしのしらせ,swarmWhen this Pokémon has 1/3 or less of its HP remaining, its Bug-type moves inflict 1.5× as much regular damage.

Overworld: If the lead Pokémon has this ability, the wild encounter rate is increased.
add スナイパー,sniperThis Pokémon infli